# tw-market-data-lake — 缺棒證據帳本（Gap Evidence Ledger v0）＋ API 抽樣比對

**A 部分（第 1–6 格）：唯讀，不呼叫任何 API、不需要憑證。**
- 逐日產生應有的 5 分格點，每個格點只有一個狀態：真實 K 棒／富邦確定缺漏／處置分盤／未解決。
- 用 FinLab 日成交股數做逐日「成交量守恆」檢查，回答缺棒裡是否藏有成交。
- 用跨標的、跨日期的訊號偵測程式邏輯遺漏。

結果先寫到 Colab 本機，再複製到 Drive `_tw-market-data-lake/audits/<ledger_id>/`（新資料夾）。全量約 40–70 分鐘。

**B 部分（第 7–9 格，選用）：唯讀的 API 抽樣比對。**
- 分層抽約 130 個股票日，加上 15 個指定日，向富邦歷史 K 線重新查詢，逐格點比對已存的 K 棒。
- 只呼叫行情端點，每個股票日 1 次請求，約 3 分鐘；每個回應都存成 JSON 並記 SHA-256。
- 不寫入任何資料集。

In [ ]:
# 1. 參數
SYMBOLS = ""              # 空白 = 全部；例 "2330,0050" 先試跑
MAX_PARTITIONS = None
COPY_TO_DRIVE = True
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
OUT_DIR = "/content/tw_lake_gap_ledger"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 4. 依賴
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)

In [ ]:
# 5. 建立缺棒證據帳本（唯讀，即時顯示進度）
import os, sys, glob, shutil, subprocess
args = [sys.executable, "-m", "tw_lake.audit.gap_ledger", "--data-root", DATA_ROOT, "--out-dir", OUT_DIR]
if SYMBOLS: args += ["--symbols", SYMBOLS]
if MAX_PARTITIONS: args += ["--max-partitions", str(MAX_PARTITIONS)]
env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
proc = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit", proc.wait())
LEDGER_DIR = sorted(glob.glob(f"{OUT_DIR}/gap_ledger_*"))[-1]
if COPY_TO_DRIVE:
    dest = f"{DATA_ROOT}/_tw-market-data-lake/audits/{os.path.basename(LEDGER_DIR)}"
    shutil.copytree(LEDGER_DIR, dest)
    print("copied to", dest)

In [ ]:
# 6. 摘要
from IPython.display import Markdown, display
display(Markdown(open(f"{LEDGER_DIR}/gap_ledger_summary.md", encoding="utf-8").read()))

## B. API 抽樣比對（選用，唯讀）
需要 Colab Secrets（與 Runner 相同）和富邦 SDK。只呼叫歷史 K 線行情端點，不會呼叫下單或帳務。

In [ ]:
# 7. Secrets + 富邦 SDK（值不會被印出）
from google.colab import userdata
import os, subprocess, sys
FUBON_PFX_DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/Fubon_Cert/your_cert.pfx"   # 改成實際位置
for k, req in (("FUBON_USER_ID", True), ("FUBON_PASSWORD", False), ("FUBON_CERT_PASSWORD", False), ("FUBON_API_KEY", False)):
    try:
        v = userdata.get(k)
    except Exception:
        v = None
    if req and not v:
        raise RuntimeError(f"缺少 Colab Secret: {k}")
    os.environ[k] = v or ""
os.environ["FUBON_CERT_PATH"] = FUBON_PFX_DRIVE_PATH
subprocess.run([sys.executable, f"{PROJECT_DIR}/scripts/install_colab_dependencies.py"], check=True)

In [ ]:
# 8. 抽樣比對
PER_STRATUM = 25
EXTRA = ("00720B:2025-06-23,1225:2023-08-17,2073:2024-12-16,3226:2024-08-22,3454:2026-03-30,4716:2025-06-23,"
         "5013:2024-12-19,5278:2023-10-24,5455:2026-04-08,6021:2023-09-07,6021:2023-12-28,6144:2026-06-23,"
         "6661:2026-10-02,6904:2026-03-26,6949:2024-12-02")
SC_OUT = "/content/tw_lake_spotcheck"
args = [sys.executable, "-m", "tw_lake.audit.api_spotcheck", "--data-root", DATA_ROOT, "--ledger-dir", LEDGER_DIR,
        "--out-dir", SC_OUT, "--per-stratum", str(PER_STRATUM), "--extra", EXTRA]
env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
proc = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit", proc.wait())

In [ ]:
# 9. 複製抽樣結果到 Drive，並等待上傳
import glob, shutil
SC_DIR = sorted(glob.glob(f"{SC_OUT}/spotcheck_*"))[-1]
dest = f"{LEDGER_DIR.replace(OUT_DIR, DATA_ROOT + '/_tw-market-data-lake/audits')}/{os.path.basename(SC_DIR)}"
shutil.copytree(SC_DIR, dest)
print("copied to", dest)
drive.flush_and_unmount()
print("Drive flushed")